<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/Quantum_Physics_Beyond_Standard_Model_Feynman_Diagrams.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Beyond the Standard Model: Scientific Animation Project

This project provides a scientifically rigorous framework for generating a high-quality animation exploring the theoretical boundaries of modern particle physics. Using a minimalist Feynman-diagram aesthetic, the script visualizes several speculative theories that extend the current Standard Model.

## Scientific Scene Specifications

1. **Gravity vs. Electromagnetism**: A comparison between the known photon exchange and the hypothetical graviton.
2. **Dark Matter Interactions**: A visualization of potential interaction portals between dark matter and baryonic matter via unknown force carriers.
3. **Majorana Neutrinos**: An exploration of the theory that neutrinos may be their own antiparticles, potentially explaining matter-antimatter asymmetry.
4. **Proton Decay**: Depiction of Grand Unified Theory (GUT) predictions involving X Boson-mediated decay.
5. **Top Quark Dynamics**: Visualizing the decay of the heaviest known elementary particle before the onset of hadronization.
6. **Quantum Field Precision**: Demonstrating the accuracy of loop corrections and the g-factor in Quantum Field Theory.

## Technical Implementation

* **Environment**: Python 3
* **Dependencies**: Matplotlib (rendering), NumPy (mathematical modeling), FFmpeg (video encoding)
* **Resolution**: 1280x720 (720p HD)
* **Framerate**: 30 FPS
* **Duration**: 54 Seconds

---
**Author:** Mugambi Ndwiga | **Social:** @craftsandengineering

In [ ]:
"""
Beyond the Standard Model — Scientific Animation

Author:
Mugambi Ndwiga

Instagram:
@craftsandengineering

Description:
High-fidelity educational scientific animation exploring
major open problems beyond the Standard Model of particle physics.

Topics:
- Quantum gravity & hypothetical gravitons
- Dark matter portal models
- Majorana neutrinos & neutrinoless double beta decay
- Proton decay in Grand Unified Theories
- Top quark decay and Higgs coupling
- Quantum vacuum fluctuations and precision QED

Platform:
Google Colab

Libraries:
matplotlib, numpy

"""

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FFMpegWriter
import matplotlib.patches as patches
from matplotlib.patheffects import withStroke
from IPython.display import HTML, display
from base64 import b64encode
import google.colab.files

# =========================================================
# GLOBAL SETTINGS
# =========================================================

FPS = 30
DURATION = 54
TOTAL_FRAMES = FPS * DURATION

WIDTH, HEIGHT = 1280, 720

BG_COLOR = '#02030a'
TEXT_COLOR = '#f5f7fa'

ELECTRON_COLOR = '#42a5f5'
PHOTON_COLOR = '#ffee58'
GLUON_COLOR = '#66bb6a'
QUARK_COLOR = '#ef5350'
GRAVITON_COLOR = '#cfd8dc'
DARK_COLOR = '#90a4ae'
MEDIATOR_COLOR = '#ff4081'
WARNING_COLOR = '#ff5252'
HIGGS_COLOR = '#ab47bc'
NEUTRINO_COLOR = '#80deea'

plt.rcParams['font.family'] = 'DejaVu Sans'

# =========================================================
# HELPER FUNCTIONS
# =========================================================

def ease(t):
    return 0.5 - 0.5 * np.cos(np.pi * t)

def fade_alpha(t):
    alpha = 1.0
    if t < 0.06:
        alpha = ease(t / 0.06)
    elif t > 0.94:
        alpha = ease((1 - t) / 0.06)
    return alpha

def glow_text(ax, x, y, text, color, size=14, **kwargs):
    txt = ax.text(
        x, y, text,
        color=color,
        fontsize=size,
        **kwargs
    )
    txt.set_path_effects([
        withStroke(linewidth=2, foreground='black')
    ])
    return txt

def wavy_line(p1, p2, amp=0.08, wavelength=0.35, phase=0):
    dist = np.linalg.norm(p2 - p1)
    n = max(int(dist * 80), 20)

    t = np.linspace(0, 1, n)
    line = np.outer(1 - t, p1) + np.outer(t, p2)

    perp = np.array([-(p2[1] - p1[1]), p2[0] - p1[0]], dtype=float)
    perp /= np.linalg.norm(perp) + 1e-9

    wave = amp * np.sin(2 * np.pi * dist * t / wavelength + phase)

    return line + np.outer(wave, perp)

def zigzag_line(p1, p2, amp=0.08, count=8):
    t = np.linspace(0, 1, count * 2 + 1)

    line = np.outer(1 - t, p1) + np.outer(t, p2)

    perp = np.array([-(p2[1] - p1[1]), p2[0] - p1[0]], dtype=float)
    perp /= np.linalg.norm(perp) + 1e-9

    offsets = np.zeros_like(t)
    offsets[1::2] = amp
    offsets[2::2] = -amp

    return line + np.outer(offsets, perp)

def draw_particle_line(ax, p1, p2, color, lw=2, alpha=1):
    ax.plot([p1[0], p2[0]],
            [p1[1], p2[1]],
            color=color,
            lw=lw,
            alpha=alpha)

def draw_wave(ax, p1, p2, color, phase=0, lw=2, alpha=1):
    pts = wavy_line(np.array(p1), np.array(p2), phase=phase)
    ax.plot(pts[:,0], pts[:,1],
            color=color,
            lw=lw,
            alpha=alpha)

def draw_gluon(ax, p1, p2, color, alpha=1):
    pts = zigzag_line(np.array(p1), np.array(p2))
    ax.plot(pts[:,0], pts[:,1],
            color=color,
            lw=2,
            alpha=alpha)

def draw_vertex(ax, pos, t, alpha):
    pulse = 0.06 + 0.04 * np.sin(t * 12)
    ax.add_patch(
        plt.Circle(
            pos,
            pulse,
            color='white',
            alpha=0.8 * alpha
        )
    )

def add_experiment_label(ax, label, alpha):
    glow_text(
        ax,
        0,
        -2.55,
        label,
        '#bbbbbb',
        size=10,
        ha='center',
        alpha=alpha * 0.8
    )

# =========================================================
# FIGURE SETUP
# =========================================================

fig, ax = plt.subplots(figsize=(12.8, 7.2), dpi=100)

fig.patch.set_facecolor(BG_COLOR)

watermark_text = "Mugambi Ndwiga | @craftsandengineering"

# =========================================================
# SCENE 1
# =========================================================

def scene_1(ax, t, alpha):

    phase = t * 20

    glow_text(
        ax, 0, 3.05,
        "Quantum Gravity vs Electromagnetism",
        TEXT_COLOR,
        size=20,
        ha='center',
        alpha=alpha
    )

    # Electromagnetic interaction
    draw_particle_line(ax, (-4, 1.2), (-2.6, 0.5), ELECTRON_COLOR, alpha=alpha)
    draw_particle_line(ax, (-4, -0.2), (-2.6, 0.5), ELECTRON_COLOR, alpha=alpha)

    draw_wave(ax, (-2.6, 0.5), (-1.0, 0.5),
              PHOTON_COLOR,
              phase=phase,
              alpha=alpha)

    draw_particle_line(ax, (-1.0, 0.5), (0.4, 1.2), ELECTRON_COLOR, alpha=alpha)
    draw_particle_line(ax, (-1.0, 0.5), (0.4, -0.2), ELECTRON_COLOR, alpha=alpha)

    draw_vertex(ax, (-2.6, 0.5), t, alpha)
    draw_vertex(ax, (-1.0, 0.5), t, alpha)

    glow_text(ax, -1.8, 0.9, "Photon \u03b3", PHOTON_COLOR,
              size=11, ha='center', alpha=alpha)

    glow_text(ax, -2.0, -1.5,
              "Observed Quantum Force",
              TEXT_COLOR,
              size=11,
              ha='center',
              alpha=alpha)

    # Gravity
    draw_particle_line(ax, (1.3, 1.0), (2.5, 0.4),
                       GRAVITON_COLOR,
                       alpha=alpha * 0.6)

    draw_particle_line(ax, (1.3, -0.2), (2.5, 0.4),
                       GRAVITON_COLOR,
                       alpha=alpha * 0.6)

    grav = wavy_line(np.array([2.5, 0.4]),
                     np.array([4.0, 0.4]),
                     amp=0.03,
                     wavelength=0.18,
                     phase=phase)

    ax.plot(
        grav[:,0],
        grav[:,1],
        color=GRAVITON_COLOR,
        lw=1.5,
        alpha=alpha * 0.7
    )

    glow_text(ax, 3.2, 0.8,
              "Hypothetical Graviton",
              GRAVITON_COLOR,
              size=10,
              ha='center',
              alpha=alpha)

    glow_text(ax, 2.8, -1.5,
              "Gravity is ~10\u2074\u2070 times weaker",
              TEXT_COLOR,
              size=10,
              ha='center',
              alpha=alpha)

    glow_text(
        ax,
        0,
        -3.0,
        "No quantum particle of gravity has ever been experimentally detected",
        TEXT_COLOR,
        size=11,
        ha='center',
        alpha=alpha
    )

# =========================================================
# SCENE 2
# =========================================================

def scene_2(ax, t, alpha):

    phase = t * 20

    glow_text(
        ax, 0, 3.05,
        "Dark Matter Portal Models",
        TEXT_COLOR,
        size=20,
        ha='center',
        alpha=alpha
    )

    # dark matter
    for y in [1.0, -1.0]:
        draw_particle_line(
            ax,
            (-4, y),
            (-1.8, 0),
            DARK_COLOR,
            lw=2,
            alpha=alpha
        )

    draw_vertex(ax, (-1.8, 0), t, alpha)

    draw_wave(
        ax,
        (-1.8, 0),
        (1.8, 0),
        MEDIATOR_COLOR,
        phase=phase,
        lw=2.5,
        alpha=alpha
    )

    glow_text(
        ax,
        0,
        0.45,
        "Dark Photon / Z\u2032 Portal",
        MEDIATOR_COLOR,
        size=10,
        ha='center',
        alpha=alpha
    )

    draw_vertex(ax, (1.8, 0), t, alpha)

    for y in [1.0, -1.0]:
        draw_particle_line(
            ax,
            (1.8, 0),
            (4, y),
            QUARK_COLOR,
            lw=2,
            alpha=alpha
        )

    glow_text(
        ax,
        -3.2,
        1.4,
        "Dark Matter \u03c7",
        DARK_COLOR,
        size=11,
        alpha=alpha
    )

    glow_text(
        ax,
        3.0,
        1.4,
        "Standard Model Quarks",
        QUARK_COLOR,
        size=11,
        alpha=alpha
    )

    add_experiment_label(
        ax,
        "Searches: LHC \u2022 Xenon1T \u2022 LUX-ZEPLIN",
        alpha
    )

    glow_text(
        ax,
        0,
        -3.0,
        "Dark matter appears gravitationally, but barely interacts with light",
        TEXT_COLOR,
        size=11,
        ha='center',
        alpha=alpha
    )

# =========================================================
# SCENE 3
# =========================================================

def scene_3(ax, t, alpha):

    phase = t * 10

    glow_text(
        ax, 0, 3.05,
        "Majorana Neutrinos & Neutrinoless Double Beta Decay",
        TEXT_COLOR,
        size=18,
        ha='center',
        alpha=alpha
    )

    # nuclei
    ax.add_patch(
        plt.Circle(
            (-2.8, 0),
            0.45,
            color='#546e7a',
            alpha=0.5 * alpha
        )
    )

    ax.add_patch(
        plt.Circle(
            (2.8, 0),
            0.45,
            color='#546e7a',
            alpha=0.5 * alpha
        )
    )

    # W bosons
    draw_wave(ax,
              (-2.2, 0),
              (-0.7, 0.8),
              MEDIATOR_COLOR,
              phase=phase,
              alpha=alpha)

    draw_wave(ax,
              (0.7, 0.8),
              (2.2, 0),
              MEDIATOR_COLOR,
              phase=-phase,
              alpha=alpha)

    # Majorana neutrino internal line
    ax.plot(
        [-0.7, 0.7],
        [0.8, 0.8],
        color=NEUTRINO_COLOR,
        lw=2,
        ls='--',
        alpha=alpha
    )

    glow_text(
        ax,
        0,
        1.15,
        "\u03bd = anti-\u03bd ?",
        NEUTRINO_COLOR,
        size=12,
        ha='center',
        alpha=alpha
    )

    # electrons emitted
    draw_particle_line(
        ax,
        (-0.7, 0.8),
        (-1.6, 2.0),
        ELECTRON_COLOR,
        alpha=alpha
    )

    draw_particle_line(
        ax,
        (0.7, 0.8),
        (1.6, 2.0),
        ELECTRON_COLOR,
        alpha=alpha
    )

    glow_text(ax, -1.9, 2.1, "e\u207b", ELECTRON_COLOR,
              size=12, alpha=alpha)

    glow_text(ax, 1.8, 2.1, "e\u207b", ELECTRON_COLOR,
              size=12, alpha=alpha)

    add_experiment_label(
        ax,
        "Searches: GERDA \u2022 KamLAND-Zen \u2022 LEGEND",
        alpha
    )

    glow_text(
        ax,
        0,
        -3.0,
        "If neutrinos are their own antiparticles, lepton number may not be conserved",
        TEXT_COLOR,
        size=11,
        ha='center',
        alpha=alpha
    )

# =========================================================
# SCENE 4
# =========================================================

def scene_4(ax, t, alpha):

    phase = t * 16

    glow_text(
        ax, 0, 3.05,
        "Proton Decay in Grand Unified Theories",
        TEXT_COLOR,
        size=19,
        ha='center',
        alpha=alpha
    )

    # proton
    proton = plt.Circle(
        (-3.0, 0),
        0.7,
        edgecolor='#90caf9',
        facecolor=(0.3,0.5,1,0.08),
        lw=2,
        alpha=alpha
    )
    ax.add_patch(proton)

    glow_text(
        ax,
        -3.0,
        0,
        "Proton",
        ELECTRON_COLOR,
        size=15,
        ha='center',
        va='center',
        alpha=alpha
    )

    # X boson
    draw_wave(
        ax,
        (-2.3, 0),
        (-0.6, 0),
        WARNING_COLOR,
        phase=phase,
        alpha=alpha
    )

    glow_text(
        ax,
        -1.45,
        0.45,
        "X Boson",
        WARNING_COLOR,
        size=11,
        ha='center',
        alpha=alpha
    )

    # decay products
    draw_particle_line(
        ax,
        (-0.6, 0),
        (2.0, 1.2),
        ELECTRON_COLOR,
        alpha=alpha
    )

    glow_text(
        ax,
        2.2,
        1.3,
        "e\u207a",
        ELECTRON_COLOR,
        size=13,
        alpha=alpha
    )

    photon = wavy_line(
        np.array([-0.6, 0]),
        np.array([2.0, -1.2]),
        phase=phase
    )

    ax.plot(
        photon[:,0],
        photon[:,1],
        color=PHOTON_COLOR,
        lw=2,
        alpha=alpha
    )

    glow_text(
        ax,
        2.2,
        -1.3,
        "\u03c0\u2070",
        PHOTON_COLOR,
        size=13,
        alpha=alpha
    )

    add_experiment_label(
        ax,
        "Limits from Super-Kamiokande: lifetime > 10\u2073\u2074 years",
        alpha
    )

    glow_text(
        ax,
        0,
        -3.0,
        "Some Grand Unified Theories predict that all matter is ultimately unstable",
        TEXT_COLOR,
        size=11,
        ha='center',
        alpha=alpha
    )

# =========================================================
# SCENE 5
# =========================================================

def scene_5(ax, t, alpha):

    phase = t * 18

    glow_text(
        ax, 0, 3.05,
        "Top Quark & the Higgs Field",
        TEXT_COLOR,
        size=20,
        ha='center',
        alpha=alpha
    )

    draw_gluon(
        ax,
        (-4, 1),
        (-2, 0),
        GLUON_COLOR,
        alpha=alpha
    )

    draw_gluon(
        ax,
        (-4, -1),
        (-2, 0),
        GLUON_COLOR,
        alpha=alpha
    )

    draw_vertex(ax, (-2, 0), t, alpha)

    draw_particle_line(
        ax,
        (-2, 0),
        (0.3, 0),
        QUARK_COLOR,
        lw=4,
        alpha=alpha
    )

    glow_text(
        ax,
        -0.7,
        0.4,
        "Top Quark (t)",
        QUARK_COLOR,
        size=13,
        ha='center',
        alpha=alpha
    )

    # decay
    draw_particle_line(
        ax,
        (0.3, 0),
        (2.5, 1.3),
        QUARK_COLOR,
        alpha=alpha
    )

    glow_text(
        ax,
        2.8,
        1.4,
        "Bottom Quark (b)",
        QUARK_COLOR,
        size=11,
        alpha=alpha
    )

    draw_wave(
        ax,
        (0.3, 0),
        (2.5, -1.3),
        MEDIATOR_COLOR,
        phase=phase,
        alpha=alpha
    )

    glow_text(
        ax,
        2.8,
        -1.4,
        "W\u207a Boson",
        MEDIATOR_COLOR,
        size=11,
        alpha=alpha
    )

    # Higgs coupling pulse
    pulse = 0.2 + 0.1 * np.sin(t * 18)

    higgs = plt.Circle(
        (-0.7, -1.2),
        0.45 + pulse,
        color=HIGGS_COLOR,
        alpha=0.15 * alpha
    )

    ax.add_patch(higgs)

    glow_text(
        ax,
        -0.7,
        -1.2,
        "Strong Higgs\nCoupling",
        HIGGS_COLOR,
        size=10,
        ha='center',
        va='center',
        alpha=alpha
    )

    add_experiment_label(
        ax,
        "Produced at CERN's Large Hadron Collider",
        alpha
    )

    glow_text(
        ax,
        0,
        -3.0,
        "The top quark is the heaviest known elementary particle",
        TEXT_COLOR,
        size=11,
        ha='center',
        alpha=alpha
    )

# =========================================================
# SCENE 6
# =========================================================

def scene_6(ax, t, alpha):

    phase = t * 24

    glow_text(
        ax, 0, 3.05,
        "Quantum Vacuum Fluctuations & Precision QED",
        TEXT_COLOR,
        size=18,
        ha='center',
        alpha=alpha
    )

    draw_particle_line(
        ax,
        (-4, 0),
        (-1.5, 0),
        ELECTRON_COLOR,
        alpha=alpha
    )

    draw_particle_line(
        ax,
        (1.5, 0),
        (4, 0),
        ELECTRON_COLOR,
        alpha=alpha
    )

    theta = np.linspace(0, 2*np.pi, 200)

    r = 1.2 + 0.08*np.sin(theta*8 + phase)

    x = r*np.cos(theta)
    y = r*np.sin(theta)

    ax.plot(
        x,
        y,
        color=NEUTRINO_COLOR,
        lw=2,
        alpha=0.6 * alpha
    )

    # vacuum loops
    for angle in np.linspace(0, 2*np.pi, 8, endpoint=False):

        x0 = 0.75*np.cos(angle)
        y0 = 0.75*np.sin(angle)

        loop = plt.Circle(
            (x0, y0),
            0.18 + 0.03*np.sin(phase + angle*4),
            edgecolor=PHOTON_COLOR,
            facecolor='none',
            lw=1,
            alpha=0.5 * alpha
        )

        ax.add_patch(loop)

    glow_text(
        ax,
        0,
        0,
        "Quantum Vacuum",
        TEXT_COLOR,
        size=13,
        ha='center',
        alpha=alpha
    )

    if t > 0.55:

        glow_text(
            ax,
            0,
            -1.8,
            "Electron g-factor \u2248 2.00231930436",
            TEXT_COLOR,
            size=15,
            ha='center',
            alpha=alpha
        )

        glow_text(
            ax,
            0,
            -2.2,
            "Theory and experiment agree to ~12 decimal places",
            PHOTON_COLOR,
            size=10,
            ha='center',
            alpha=alpha
        )

    add_experiment_label(
        ax,
        "Quantum Electrodynamics is the most precise theory ever tested",
        alpha
    )

    glow_text(
        ax,
        0,
        -3.0,
        "Even empty space contains fluctuating quantum fields",
        TEXT_COLOR,
        size=11,
        ha='center',
        alpha=alpha
    )

# =========================================================
# MAIN UPDATE FUNCTION
# =========================================================

def update(frame):

    ax.clear()

    ax.set_facecolor(BG_COLOR)

    ax.set_xlim(-4.5, 4.5)
    ax.set_ylim(-3.5, 3.5)

    ax.axis('off')

    glow_text(
        ax,
        4.35,
        -3.3,
        watermark_text,
        TEXT_COLOR,
        size=9,
        ha='right',
        alpha=0.25
    )

    scene_duration = TOTAL_FRAMES // 6

    scene_idx = min(frame // scene_duration, 5)

    t_scene = (frame % scene_duration) / scene_duration

    alpha = fade_alpha(t_scene)

    if scene_idx == 0:
        scene_1(ax, t_scene, alpha)

    elif scene_idx == 1:
        scene_2(ax, t_scene, alpha)

    elif scene_idx == 2:
        scene_3(ax, t_scene, alpha)

    elif scene_idx == 3:
        scene_4(ax, t_scene, alpha)

    elif scene_idx == 4:
        scene_5(ax, t_scene, alpha)

    elif scene_idx == 5:
        scene_6(ax, t_scene, alpha)

# =========================================================
# RENDER
# =========================================================

writer = FFMpegWriter(
    fps=FPS,
    bitrate=3200
)

output_file = "beyond_standard_model_enhanced.mp4"

with writer.saving(fig, output_file, 100):

    for i in range(TOTAL_FRAMES):

        update(i)

        writer.grab_frame()

plt.close(fig)

# =========================================================
# DISPLAY
# =========================================================

mp4 = open(output_file, 'rb').read()

data_url = (
    "data:video/mp4;base64," +
    b64encode(mp4).decode()
)

display(
    HTML(
        f"""
        <video width=960 controls autoplay loop>
            <source src="{data_url}" type="video/mp4">
        </video>
        """
    )
)

google.colab.files.download(output_file)

SyntaxError: unterminated string literal (detected at line 291) (3819280065.py, line 291)